# 离线 IQL vs 在线 PPO 对比(proactive RR / D64)

**目的**:奖励已改成**有界固定搁浅罚 K=50**(砍掉到地平线无界项的方差,让 critic 学得动)。在此基础上,**同一 val 集**上对比:**离线 IQL** vs **在线 PPO(有界奖励 + 松 KL + curriculum)**,看哪条路更适合我们。

**中立预期**(D64):主问题是奖励(已修)+ 任务结构,不是算法之争;两者都可能不够,用**实验**分高下。IQL 局限:数据里没有"从坏状态恢复"的样本 → 它学的是**避免**进坏状态;恢复留给后续 **AWAC**(用 IQL 策略热启动 PPO)。


## 设置


In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## ① 生成离线转移数据集(**本地已生成+拷贝的话,跳过本格**)

你在本地跑了 `scripts/33 ... 5000` 并把 `artifacts/dataset/offline_shards_train/` 拷到服务器 → 本格可跳过(就算跑,幂等会显示「已存跳过 120」)。上限默认已是 5000(信号员≤3848 不受影响,规则基线 gridlock 局截到 5000)。


In [ ]:
!{PREFIX} scripts/33_build_offline_dataset.py train 24 70   # 24天 × 5策略, 70 核并行, 分片+进度+续跑


## ② 训练 IQL(离线)
actor 复用 PanelOptionPolicy(评估口径同 PPO),可选热启动。重跑本格=断点续训。


In [ ]:
!{PREFIX} scripts/34_train_iql.py \
    --dataset artifacts/dataset/offline_shards_train \
    --encoder gnn --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --iters 3000 --batch 128 --eval-every 150 --gamma 0.99 --tau 0.7 --beta 3.0 \
    --run-tag iql_gnn_s0 --resume


## ③ 训练 PPO(在线,有界奖励 + 松 KL β=0.03 + curriculum)
奖励有界(默认)、KL 松、curriculum 由易到难、mastery-only 升档。重跑本格=断点续训。


In [ ]:
!{PREFIX} scripts/13_train_ppo.py --encoder gnn \
    --bc-ckpt artifacts/dataset/bc_fullpanel_gnn.pt \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 --promote-at 0.9 \
    --train-eval-days 6 --iters 80 --steps-per-iter 2048 --eval-every 5 \
    --early-stop-patience 6 --run-tag ppo_cur_gnn_s0 --resume


## ④ 对比 IQL vs PPO(同一 val 集 + 参照线)


In [ ]:
import json, torch
from pathlib import Path
rep=Path('artifacts/reports')
def load(tag):
    if (rep/f'{tag}.json').exists(): d=json.loads((rep/f'{tag}.json').read_text())
    elif (rep/f'{tag}_last.pt').exists(): d=torch.load(rep/f'{tag}_last.pt',map_location='cpu',weights_only=False)
    else: return None
    return d
iql=load('iql_gnn_s0'); ppo=load('ppo_cur_gnn_s0')
base=(ppo or iql or {}).get('baselines',{})
print(f"{'方法':<10} {'val完成率':>9} {'val_reward':>12} {'best@it':>8}")
for name,d in [('IQL',iql),('PPO',ppo)]:
    if d and d.get('best'): b=d['best']; print(f"{name:<10} {b.get('completion'):>9} {b.get('reward'):>12.0f} {b.get('iter'):>8}")
for k in ['signaller_replay','lookup_most_frequent','greedy_set']:
    if k in base: print(f"{k:<10} {base[k]['eval_completion_rate']:>9.3f} {base[k]['eval_episode_reward']:>12.0f}")
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    plt.figure(figsize=(8,3))
    for name,d in [('IQL',iql),('PPO',ppo)]:
        if d and d.get('evals'):
            ev=d['evals']; plt.plot([e['iter'] for e in ev],[e['eval_completion_rate'] for e in ev],'-o',label=name)
    if 'signaller_replay' in base: plt.axhline(1.0,ls='--',c='g',lw=0.8,label='signaller=1.0')
    plt.ylim(0,1.05); plt.xlabel('iter(注:IQL/PPO 尺度不同)'); plt.ylabel('val 完成率'); plt.legend(); plt.tight_layout()
    plt.savefig(rep/'iql_vs_ppo.png',dpi=110); plt.show()
except Exception as ex: print('plot skipped:',ex)


## 说明 / 下一步
- **若 IQL 明显好过 PPO**:说明离线价值学习对我们更对症;接着可上 **AWAC**(用 IQL 的 best 权重热启动 PPO 的 §③,在线补"恢复"样本)。
- **若两者都上不去**:坐实"问题在任务结构"→ 下一步动**任务结构**(搁浅/入区堆积 game over,把 episode 变短变干净)。
- test 全程封存(D43),headline 留到最后。
